In [85]:
import pandas as pd
from tqdm.auto import tqdm
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
from matplotlib.colors import to_rgba

tqdm.pandas()

In [43]:
high_pop_songs = pd.read_csv("../data/high_popularity_spotify_data.csv")
low_pop_songs = pd.read_csv("../data/low_popularity_spotify_data.csv")
high_pop_songs.head()

,energy,tempo,danceability,playlist_genre,loudness,liveness,valence,track_artist,time_signature,speechiness,...,instrumentalness,track_album_id,mode,key,duration_ms,acousticness,id,playlist_subgenre,type,playlist_id
0,0.592,157.969,0.521,pop,-7.777,0.122,0.535,"Lady Gaga, Bruno Mars",3,0.0304,...,0.0000,10FLjwfpbxLmW8c25Xyc2N,0,6,251668,0.3080,2plbrEY59IikOBgBGLjaoe,mainstream,audio_features,37i9dQZF1DXcBWIGoYBM5M
1,0.507,104.978,0.747,pop,-10.171,0.117,0.438,Billie Eilish,4,0.0358,...,0.0608,7aJuG4TFXa2hmE4z1yxc3n,1,2,210373,0.2000,6dOtVTDdiauQNBQEDOtlAB,mainstream,audio_features,37i9dQZF1DXcBWIGoYBM5M
2,0.808,108.548,0.554,pop,-4.169,0.159,0.372,Gracie Abrams,4,0.0368,...,0.0000,0hBRqPYPXhr1RkTDG3n4Mk,1,1,166300,0.2140,7ne4VBA60CxGM75vw0EYad,mainstream,audio_features,37i9dQZF1DXcBWIGoYBM5M
3,0.910,112.966,0.670,pop,-4.070,0.304,0.786,Sabrina Carpenter,4,0.0634,...,0.0000,4B4Elma4nNDUyl6D5PvQkj,0,0,157280,0.0939,1d7Ptw3qYcfpdLNL5REhtJ,mainstream,audio_features,37i9dQZF1DXcBWIGoYBM5M
4,0.783,149.027,0.777,pop,-4.477,0.355,0.939,"ROSÉ, Bruno Mars",4,0.2600,...,0.0000,2IYQwwgxgOIn7t3iF6ufFD,0,0,169917,0.0283,5vNRhkKd0yEAg8suGBpjeY,mainstream,audio_features,37i9dQZF1DXcBWIGoYBM5M


# Next we'll look at any missing values and decide what to do with those.

In [44]:
print(f"{(low_pop_songs.loc[low_pop_songs.isna().any(axis=1)].shape[0] / low_pop_songs.shape[0])*100:.3f}% of low popularity songs have cols with a missing value")
display(low_pop_songs.loc[low_pop_songs.isna().any(axis=1)])

0.032% of low popularity songs have cols with a missing value


,time_signature,track_popularity,speechiness,danceability,playlist_name,track_artist,duration_ms,energy,playlist_genre,playlist_subgenre,...,track_album_id,playlist_id,track_id,valence,key,tempo,loudness,acousticness,liveness,track_album_name
263,NaN,50,NaN,NaN,Chill Lofi Study Beats,Berhanio,NaN,NaN,lofi,chill,...,4X9vWjNWlZ3qAtyDZ2ZhDj,37i9dQZF1DX8Uebhn9wzrS,2teI76KKFE6qkpLZJs7tZ7,NaN,NaN,NaN,NaN,NaN,NaN,Make It


We have one row with a na value in it or less than a percent of the low popularity songs, have a column with a missing value. So we will just drop this single row.

In [45]:
print(f"{(high_pop_songs.loc[high_pop_songs.isna().any(axis=1)].shape[0] / high_pop_songs.shape[0])*100:.3f}% of high popularity songs have cols with a missing value")
display(high_pop_songs.loc[high_pop_songs.isna().any(axis=1)].drop(columns=['energy','tempo','danceability','playlist_genre'])) # Dropped these columns so you can see the NA value col.

0.059% of high popularity songs have cols with a missing value


,loudness,liveness,valence,track_artist,time_signature,speechiness,track_popularity,track_href,uri,track_album_name,...,instrumentalness,track_album_id,mode,key,duration_ms,acousticness,id,playlist_subgenre,type,playlist_id
665,-2.515,0.19,0.86,NAYEON,4,0.0607,73,https://api.spotify.com/v1/tracks/0V2passWyAXn...,spotify:track:0V2passWyAXnON67kfAj7y,NaN,...,0.0,5zQI9dFbS9TrhvC9clgjz7,0,3,162840,0.0404,0V2passWyAXnON67kfAj7y,modern,audio_features,1bNuDsel3P60p11Z7vfHMR


We also only have a single row that makes up less than a percent of the overall high popularity tracks, so we don't really loose much by dropping this single column as well. We could probably impute these missing values but since the NA is just in a single row it is not really worth the effort.

### So for addressing the missing values the most suitable choice given the small quantity of rows affected is to just drop the rows with any missing values.

In [46]:
prev_low_shape = low_pop_songs.shape[0]
low_pop_songs = low_pop_songs.dropna()
print(f"In low pop songs there were {prev_low_shape-low_pop_songs.shape[0]} rows with na values droped")

In low pop songs there were 1 rows with na values droped


In [47]:
prev_high_shape = high_pop_songs.shape[0]
high_pop_songs = high_pop_songs.dropna()
print(f"In high pop songs there were {prev_high_shape-high_pop_songs.shape[0]} rows with na values droped")

In high pop songs there were 1 rows with na values droped


# Now another cleaning aspect of this initial data is that we found a column with a single value for all entries. Since there is no variation between entries this col doesn't really tell us anything useful so it should be dropped. The way we will drop it is by just not including it in the selected columns in a later step.

In [48]:
print(f"The unique values in type col for low popularity songs is: {list(low_pop_songs['type'].unique())}")
print(f"The unique values in type col for high popularity songs is: {list(high_pop_songs['type'].unique())}")

The unique values in type col for low popularity songs is: ['audio_features']
The unique values in type col for high popularity songs is: ['audio_features']


# Now get the first artist from every list of artists for each track. The reason for doing this is to make this column flat and later analysis easier.

In [49]:
# For the sake of being better able to draw comparisons with this dataset we are only going to keep the first artist listed for each song.
def get_firstlisted_artist(artist_list):
    try:
        return(artist_list.split(",")[0].strip())
    except Exception:
        return np.nan

high_pop_songs['first_listed_artist'] = high_pop_songs['track_artist'].progress_apply(get_firstlisted_artist)
low_pop_songs['first_listed_artist'] = low_pop_songs['track_artist'].progress_apply(get_firstlisted_artist)

100%|██████████| 3144/3144 [00:00<00:00, 1456793.17it/s]


# Also, for the 'mode' column, since spotify api describes it as the following "indicates the modality (major or minor) of a track, the type of scale from which its melodic content is derived. Major is represented by 1 and minor is 0." we'll turn that col into a string with the fully written out meaning.

Keeping the column as one-hot encoded would be needed if we were planning on doing logistic regression or some other supervised learning task on this column downstream but supervised methods will not be an focus of this project.

In [50]:
high_pop_songs['mode'] = high_pop_songs['mode'].apply(lambda x: 'Major' if x ==1 else 'Minor' if x == 0 else x)
low_pop_songs['mode'] = low_pop_songs['mode'].apply(lambda x: 'Major' if x ==1 else 'Minor' if x == 0 else x)

print(f"Unique values for mode col for high pop songs is: {list(high_pop_songs['mode'].unique())}")
print(f"Unique values for mode col for low pop songs is: {list(low_pop_songs['mode'].unique())}")

Unique values for mode col for high pop songs is: ['Minor', 'Major']
Unique values for mode col for low pop songs is: ['Major', 'Minor']


# Similarly to what we did above, we are going to convert the time_signature column to more explicitly discrete bins. Again, if we were using supervised learning then performing this step would make less sense but since we are not using that method in this project, converting this to categorize should be better for interpreting properly later. 

Spotify explains this column as follows "An estimated time signature. The time signature (meter) is a notational convention to specify how many beats are in each bar (or measure). The time signature ranges from 3 to 7 indicating time signatures of "3/4", to "7/4".". However, we did have some rows with a time_signature of 1 which corresponds to an irregular time signature.


In [51]:
print(f"{'='*55} High Pop songs with irregular time signature {'='*55}")
display(high_pop_songs[high_pop_songs['time_signature'] == 1])
print(f"{'='*55} Low Pop songs with irregular time signature {'='*55}")
display(low_pop_songs[low_pop_songs['time_signature'] == 1])

======================================================= High Pop songs with irregular time signature =======================================================


,energy,tempo,danceability,playlist_genre,loudness,liveness,valence,track_artist,time_signature,speechiness,...,track_album_id,mode,key,duration_ms,acousticness,id,playlist_subgenre,type,playlist_id,first_listed_artist
190,0.68000,144.006,0.850,hip-hop,-6.180,0.0964,0.6940,Central Cee,1,0.3380,...,0aAVMtHuK9wX1mQozWvdSZ,Minor,3,173333,0.732,4vzJrkfHbCGaodPA5RY5BV,gangster,audio_features,6mODZNt8CYYEBrWAyS5vs4,Central Cee
771,0.51500,120.100,0.567,arabic,-10.265,0.0780,0.3210,Lil Tjay,1,0.2470,...,1bCBZ8LedqwVYOLDLrRbbY,Minor,6,224000,0.354,0RLLvYBopDwooCyvnTwgYu,modern,audio_features,3gZgGRvDn9tTUy8n7699pD,Lil Tjay
1445,0.86000,76.540,0.575,hip-hop,-3.165,0.0973,0.8340,Lil Yachty,1,0.2300,...,4peDphjWdwzhrTEnQbfZ4G,Minor,9,107314,0.298,1Y1jtaZOJYgXBl7lKdLc9o,chill,audio_features,37i9dQZF1EIf4OaZ1XTJYw,Lil Yachty
1497,0.00161,140.997,0.138,ambient,-43.643,0.0951,0.1550,Somnum Magnam,1,0.0532,...,0gr42nFesHmvHP2rihBldP,Major,5,144750,0.977,6KzdWw8Hi1bz1lVw2AXDzb,chill,audio_features,37i9dQZF1DX3Ogo9pFvBkY,Somnum Magnam
1548,0.51500,120.100,0.567,hip-hop,-10.265,0.0780,0.3210,Lil Tjay,1,0.2470,...,1bCBZ8LedqwVYOLDLrRbbY,Minor,6,224000,0.354,0RLLvYBopDwooCyvnTwgYu,melodic,audio_features,37i9dQZF1EIgLJJI0PCAq9,Lil Tjay
1612,0.05820,132.614,0.191,classical,-25.398,0.0941,0.0454,Ludovico Einaudi,1,0.0578,...,0Ryad9M1b1MxSjgXdHCh1c,Major,8,357707,0.961,3weNRklVDqb4Rr5MhKBR3D,neo-classical,audio_features,37i9dQZF1EIfnkSMcirbYT,Ludovico Einaudi
1614,0.05590,95.217,0.396,classical,-20.707,0.1100,0.2780,Ben Crosland,1,0.0599,...,1HBOLQ0QoDkGDGnjhfdzPy,Minor,9,78413,0.987,1yqM5O07lL7R6UAdK5jvrR,neo-classical,audio_features,37i9dQZF1EIfnkSMcirbYT,Ben Crosland


======================================================= Low Pop songs with irregular time signature =======================================================


,time_signature,track_popularity,speechiness,danceability,playlist_name,track_artist,duration_ms,energy,playlist_genre,playlist_subgenre,...,playlist_id,track_id,valence,key,tempo,loudness,acousticness,liveness,track_album_name,first_listed_artist
28,1.0,53,0.0388,0.3410,Classical Essentials,"Franz Liszt, Lise de la Salle",357747.0,0.014200,classical,essential,...,37i9dQZF1DWWEJlAGA9gs0,3GecLjGMqzjmgkvy2v41Wf,0.0384,2.0,76.382,-28.293,0.9930,0.1130,Liszt,Franz Liszt
92,1.0,64,0.0355,0.7210,Viva Latino,"Gabito Ballesteros, Natanael Cano, Luis R Conr...",182558.0,0.821000,latin,modern,...,37i9dQZF1DX10zKzsJ2jva,2q0wKlSmIWLdpCwwKjLPxZ,0.3800,11.0,104.044,-5.695,0.1780,0.0934,Presidente,Gabito Ballesteros
203,1.0,55,0.0330,0.4000,Chill Lofi Study Beats,Dozy Duzzn,129143.0,0.148000,lofi,chill,...,37i9dQZF1DX8Uebhn9wzrS,1MRxxXsWluxrcyiyy0msBd,0.1210,9.0,97.450,-22.677,0.9320,0.0891,Napwave Nirvana,Dozy Duzzn
221,1.0,53,0.0625,0.5790,Chill Lofi Study Beats,nora haze,125625.0,0.301000,lofi,chill,...,37i9dQZF1DX8Uebhn9wzrS,5dN8phxVcvz7BAIMuI4wqv,0.2890,6.0,160.024,-17.913,0.8830,0.5980,So Very Much,nora haze
294,1.0,50,0.1690,0.5820,Chill Lofi Study Beats,Violet Blanca,121250.0,0.490000,lofi,chill,...,37i9dQZF1DX8Uebhn9wzrS,043xe58AMUGPJgCnRJynj9,0.1750,10.0,132.682,-8.651,0.7300,0.1030,Laundry Tessomancy,Violet Blanca
338,1.0,48,0.0988,0.6560,Chill Lofi Study Beats,Max Hayward,148375.0,0.197000,lofi,chill,...,37i9dQZF1DX8Uebhn9wzrS,3ctgxH0ZpE50qscnZYBY1H,0.5210,2.0,147.974,-19.201,0.8340,0.6970,big buddha,Max Hayward
888,1.0,54,0.2860,0.3010,Mandopop Hits,告五人,301827.0,0.723000,mandopop,modern,...,37i9dQZF1DXahjk49z40fT,3agPPRZPNOhbCwm2026EN6,0.3590,2.0,77.143,-6.808,0.3990,0.0851,從未見過的海,告五人
971,1.0,33,0.0654,0.7480,Lofi Hip Hop,Sarcastic Sounds,118618.0,0.338000,lofi,hip-hop,...,3BOxfexQyBa1tsEZ1tJFVQ,6qWitlJBWOQBXyumtDBZAV,0.3770,11.0,152.033,-13.947,0.0418,0.3120,I Miss You,Sarcastic Sounds
1173,1.0,32,0.0359,0.2630,Minimalism,"Caterina Barbieri, Kara-Lis Coverdale",187864.0,0.064200,classical,modern,...,37i9dQZF1EIcPn17YOI0D2,3EmimXCpHQ9PW6FN0ajDeU,0.3220,7.0,165.638,-22.824,0.9950,0.1070,Fantas Variations,Caterina Barbieri
1176,1.0,61,0.0327,0.2780,Minimalism,Tony Anderson,103091.0,0.021700,classical,modern,...,37i9dQZF1EIcPn17YOI0D2,4LurYqydfPdLc9IN7eQxTF,0.2230,9.0,104.541,-37.438,0.9800,0.0926,Nuit,Tony Anderson


The most interesting thing that immediately sticks out about these entries with irregular time signatures, with the low popularity tracks especially, is how the playlist genres are largely things like Lofi or chill and also how some of the loudness values get really close to zero. More later on why loudness values near or above 0 are interesting.

In [52]:
# For now we'll still convert the time to a more explicitly categorical string.
high_pop_songs['time_signature'] = high_pop_songs['time_signature'].apply(lambda x: x  if type(x) == str else f"{int(x)}/4" if x >= 3 else 'irregular') 
                                                                                        # The final else is just to not destroy this col if this cell is ran more then once.

low_pop_songs['time_signature'] = low_pop_songs['time_signature'].apply(lambda x: x  if type(x) == str else f"{int(x)}/4" if x >= 3 else 'irregular') 
# Now print the results.
print(f"The distinct time signatures for high pop songs are: {list(high_pop_songs['time_signature'].unique())}")
print(f"The distinct time signatures for low pop songs are: {list(low_pop_songs['time_signature'].unique())}")


The distinct time signatures for high pop songs are: ['3/4', '4/4', '5/4', 'irregular']
The distinct time signatures for low pop songs are: ['4/4', '3/4', '5/4', 'irregular']


# Next we'll turn the duration_ms columns from ms to minutes. Mostly for later visualizations. This will sacrifice a tiny amount of granularity, but that tradeoff is worth it to be more inline with the song duration in minutes that most of us normal listeners would expect.

In [56]:
if not 'duration_min' in high_pop_songs.columns:
    high_pop_songs['duration_min'] = high_pop_songs['duration_ms'].apply(lambda x: x / 60000)
if not 'duration_min' in low_pop_songs.columns:
    low_pop_songs['duration_min'] = low_pop_songs['duration_ms'].apply(lambda x: x / 60000)

print(f"Max for High pop songs minutes is {high_pop_songs['duration_min'].max()}, and Min is {high_pop_songs['duration_min'].min()}")
print(f"Max for Low pop songs minutes is {low_pop_songs['duration_min'].max()}, and Min is {low_pop_songs['duration_min'].min()}")

Max for High pop songs minutes is 9.11845, and Min is 1.0278833333333333
Max for Low pop songs minutes is 22.587666666666667, and Min is 0.5895833333333333


# Finally, before combining the two datasets, the final thing we need to clean up is the loudness column. Now according to spotify's api values for this column " typically range between -60 and 0 db". So we need to first check if we have any entries that are above that in either low or high pop frames.

In [60]:
print(f"{'='*25} For the high popularity songs {(high_pop_songs[high_pop_songs['loudness'] > 0].shape[0] / high_pop_songs.shape[0])*100:.3f}% entries above 0 dB")
display(high_pop_songs[high_pop_songs['loudness'] > 0])
print(f"{'='*25} For the low popularity songs {(low_pop_songs[low_pop_songs['loudness'] > 0].shape[0] / low_pop_songs.shape[0])*100:.3f}% entries above 0 dB")
display(low_pop_songs[low_pop_songs['loudness'] > 0])

========================= For the high popularity songs 0.119% entries above 0 dB


,energy,tempo,danceability,playlist_genre,loudness,liveness,valence,track_artist,time_signature,speechiness,...,mode,key,duration_ms,acousticness,id,playlist_subgenre,type,playlist_id,first_listed_artist,duration_min
523,0.927,109.952,0.755,gaming,0.254,0.0567,0.702,"xxanteria, isq",4/4,0.0645,...,Minor,11,115348,0.00288,4VL5XwfATZuAVTW471Wpro,modern,audio_features,37i9dQZF1DWTyiBJ6yEqeu,xxanteria,1.922467
1419,0.827,129.949,0.956,brazilian,1.295,0.4230,0.614,"Mc Delux, DJ Guih Da ZO",4/4,0.2220,...,Major,9,133846,0.13600,7snnTlaWaN39nfN1PhUaT8,funk,audio_features,6G1N7RXO7gOGfFKlRyjl8G,Mc Delux,2.230767


========================= For the low popularity songs 0.159% entries above 0 dB


,time_signature,track_popularity,speechiness,danceability,playlist_name,track_artist,duration_ms,energy,playlist_genre,playlist_subgenre,...,track_id,valence,key,tempo,loudness,acousticness,liveness,track_album_name,first_listed_artist,duration_min
1408,4/4,62,0.1010,0.845,Brazilian Funk,"$werve, TOKYOSLEEP, Mc Gw",106286.0,0.972,brazilian,funk,...,39BOpF8KvdMqUZlJyfnV34,0.881,0.0,129.967,0.914,0.44400,0.5140,TOMA TOMA TOMA,$werve,1.771433
1417,4/4,59,0.3560,0.773,Brazilian Funk,"MC Teuzin PV, MC Skcot",134737.0,0.855,brazilian,funk,...,1GcYslWGW1RMEmI0ZuZ90B,0.658,11.0,86.930,0.832,0.22300,0.1210,Eu Já Sofri por Amor Mas Não Sofro Mais (Tik Tok),MC Teuzin PV,2.245617
1444,3/4,64,0.5570,0.633,Brazilian Funk,"DJ Jeeh FDC, MC LCKaiique, MC Celo BK, DJ Mene...",182769.0,0.699,brazilian,funk,...,5vdQUAFncALEH9EQvyAqIX,0.762,11.0,83.624,0.410,0.51600,0.1750,Forte Forte de Lacoste - Pra Começar o Final d...,DJ Jeeh FDC,3.046150
1448,4/4,67,0.2390,0.910,Brazilian Funk,Seek,97846.0,0.879,brazilian,funk,...,6Seth37t5I69wFbuGFrsiG,0.954,5.0,130.006,1.318,0.14300,0.0704,REI DO BRASIL,Seek,1.630767
2328,4/4,50,0.0513,0.633,Hardstyle Bangers,Lil Texas,124541.0,0.977,electronic,hardstyle,...,08XCFa4gCODqSy6FDv2Uhs,0.404,2.0,92.500,1.075,0.00131,0.4480,Drop That,Lil Texas,2.075683


Again, since these entries are still less than a percent of the overall entries we are just going to drop them or filter them out.

In [69]:
high_pop_songs = high_pop_songs.loc[high_pop_songs['loudness'] <= 0]
low_pop_songs = low_pop_songs.loc[low_pop_songs['loudness'] <= 0]

print(f"High pop songs loudness range: [{high_pop_songs['loudness'].min()},{high_pop_songs['loudness'].max()}]")
print(f"Low pop songs loudness range: [{low_pop_songs['loudness'].min()},{low_pop_songs['loudness'].max()}]")


High pop songs loudness range: [-43.643,-0.93]
Low pop songs loudness range: [-48.069,-0.033]


# Now finally, we'll combine the two dfs and select only needed columns. The reason for combining the two df's is so that we can later bin the tracks by popularity into finer buckets then just high and low.

Here needed columns were everything but the redundant id columns. For example, since we were not planning on scrapping any more info from Spotify's API (we tried doing this and the rate limiting was a nightmare), columns like 'track_album_id' are not useful for our downstream analysis as this is just a pointer id to the album this track belongs to. We did keep track_album_name and playlist_name though so we can still group by the human interpretable version of these attributes. Also in this step we turn the album release date into an actual date time format.

***A important note for later, the date attached to each track is the album's release date. Not the tracks individual release date, although the two should be tightly grouped.***

In [71]:
# Combine the high popularity and low popularity tracks.
spotify_tracks = pd.concat([high_pop_songs,low_pop_songs])[['id','energy', 'tempo', 'danceability', 'loudness',
       'liveness', 'valence', 'time_signature', 'speechiness',
       'track_album_release_date', 'instrumentalness',
       'mode', 'key', 'duration_min', 'acousticness', 'first_listed_artist',
       'track_album_name', 'playlist_name', 'playlist_genre','playlist_subgenre',
       'track_name','track_popularity']]

# Print the dropped columns.
print(f"\n\n{'='*55} Showing columns change next {'='*55}\n")
print(f"The dropped columns were the following:\n{list(set(high_pop_songs.columns) - set(spotify_tracks.columns))}")

# Print all the retained columns.
print(f"\n\nThe columns retained for our analysis were:")
for col in spotify_tracks.columns:
    print(f"\n\t{col}")

# Drop small fraction of records that do not have a full release date.
print(f"\n\n{'='*55} Dealing with the track release date column next {'='*55}\n")
spotify_tracks['date_num_dash'] = spotify_tracks['track_album_release_date'].apply(lambda x: x.count("-"))
print(f"{(spotify_tracks[spotify_tracks['date_num_dash'] != 2].shape[0]/spotify_tracks.shape[0])*100:.3f}% of records do not have full date information")
spotify_tracks = spotify_tracks.loc[spotify_tracks['date_num_dash'] == 2]
spotify_tracks['track_album_release_date'] = pd.to_datetime(spotify_tracks['track_album_release_date'])
spotify_tracks = spotify_tracks.drop(columns=['date_num_dash'])

# Sort by popularity for easier df readability.
spotify_tracks = spotify_tracks.sort_values(by='track_popularity',ascending=False).reset_index(drop=True)




======================================================= Showing columns change next =======================================================

The dropped columns were the following:
['duration_ms', 'type', 'track_album_id', 'track_href', 'uri', 'playlist_id', 'track_id', 'analysis_url', 'track_artist']


The columns retained for our analysis were:

	id

	energy

	tempo

	danceability

	loudness

	liveness

	valence

	time_signature

	speechiness

	track_album_release_date

	instrumentalness

	mode

	key

	duration_min

	acousticness

	first_listed_artist

	track_album_name

	playlist_name

	playlist_genre

	playlist_subgenre

	track_name

	track_popularity


======================================================= Dealing with the track release date column next =======================================================

2.862% of records do not have full date information


# Double check that all the data types for all the columns actually make sense. And that all the column ranges make sense.

In [ ]:
spotify_tracks.describe().loc[['min','max']]

,energy,tempo,danceability,loudness,liveness,valence,speechiness,track_album_release_date,instrumentalness,key,duration_min,acousticness,track_popularity
min,0.000202,48.232,0.0589,-48.069,0.021,0.0296,0.0219,1954-01-01 00:00:00,0.000,0.0,0.589583,0.000004,11.0
max,0.998000,241.426,0.9790,-0.033,0.979,0.9870,0.9270,2024-11-22 00:00:00,0.991,11.0,22.587667,0.995000,100.0


Based upon the spotify API document and the min and max values seen above we are in a valid range for everything.

In [82]:
spotify_tracks.dtypes

id                                     str
energy                             float64
tempo                              float64
danceability                       float64
loudness                           float64
liveness                           float64
valence                            float64
time_signature                         str
speechiness                        float64
track_album_release_date    datetime64[us]
instrumentalness                   float64
mode                                   str
key                                float64
duration_min                       float64
acousticness                       float64
first_listed_artist                    str
track_album_name                       str
playlist_name                          str
playlist_genre                         str
playlist_subgenre                      str
track_name                             str
track_popularity                     int64
dtype: object

All the data types now make sense as well.

# Write out the cleaned csv. And display to prove that everything looks ok.

In [84]:
spotify_tracks.to_csv('spotify_tracks_pop.csv',index=False)
spotify_tracks = pd.read_csv('spotify_tracks_pop.csv')
display(pd.read_csv('spotify_tracks_pop.csv').head(5))
spotify_tracks['track_album_release_date'] = pd.to_datetime(spotify_tracks['track_album_release_date'])
spotify_tracks.shape

,id,energy,tempo,danceability,loudness,liveness,valence,time_signature,speechiness,track_album_release_date,...,key,duration_min,acousticness,first_listed_artist,track_album_name,playlist_name,playlist_genre,playlist_subgenre,track_name,track_popularity
0,2plbrEY59IikOBgBGLjaoe,0.592,157.969,0.521,-7.777,0.122,0.535,3/4,0.0304,2024-08-16,...,6.0,4.194467,0.3080,Lady Gaga,Die With A Smile,Today's Top Hits,pop,mainstream,Die With A Smile,100
1,2plbrEY59IikOBgBGLjaoe,0.592,157.969,0.521,-7.777,0.122,0.535,3/4,0.0304,2024-08-16,...,6.0,4.194467,0.3080,Lady Gaga,Die With A Smile,Global Top 50,pop,global,Die With A Smile,100
2,2plbrEY59IikOBgBGLjaoe,0.592,157.969,0.521,-7.777,0.122,0.535,3/4,0.0304,2024-08-16,...,6.0,4.194467,0.3080,Lady Gaga,Die With A Smile,Top Gaming Tracks,gaming,modern,Die With A Smile,100
3,5vNRhkKd0yEAg8suGBpjeY,0.783,149.027,0.777,-4.477,0.355,0.939,4/4,0.2600,2024-10-18,...,0.0,2.831950,0.0283,ROSÉ,APT.,Today's Top Hits,pop,mainstream,APT.,98
4,5vNRhkKd0yEAg8suGBpjeY,0.783,149.027,0.777,-4.477,0.355,0.939,4/4,0.2600,2024-10-18,...,0.0,2.831950,0.0283,ROSÉ,APT.,Global Top 50,pop,global,APT.,98


(4684, 22)

In [86]:
spotify_tracks.columns

Index(['id', 'energy', 'tempo', 'danceability', 'loudness', 'liveness',
       'valence', 'time_signature', 'speechiness', 'track_album_release_date',
       'instrumentalness', 'mode', 'key', 'duration_min', 'acousticness',
       'first_listed_artist', 'track_album_name', 'playlist_name',
       'playlist_genre', 'playlist_subgenre', 'track_name',
       'track_popularity'],
      dtype='str')